In [ ]:
# ============================================
# 07_RealTime.ipynb
# REAL-TIME DETECTION KANTUK PENGEMUDI
# ============================================

import os
import sys
import json
import time
import warnings
import numpy as np
import cv2
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
from IPython.display import display, Image, clear_output
import threading
import queue

# ============================================
# SETUP
# ============================================
warnings.filterwarnings('ignore')

print("="*60)
print("🎥 REAL-TIME DETECTION KANTUK PENGEMUDI")
print("="*60)

# ============================================
# 1. SETUP PATHS
# ============================================

project_root = Path.cwd().parent if Path.cwd().name in ['notebooks', 'notebook'] else Path.cwd()
models_dir = project_root / "models"
results_dir = project_root / "results" / "realtime"
results_dir.mkdir(parents=True, exist_ok=True)

print(f"\n[Paths]")
print(f"   - Project: {project_root}")
print(f"   - Models: {models_dir}")
print(f"   - Results: {results_dir}")

# ============================================
# 2. PILIH MODEL TERBAIK
# ============================================

print("\n[Selecting Best Model]")

# Cek model yang tersedia
available_models = {}
model_info = {
    'yolo': {
        'name': 'YOLOv8',
        'path': models_dir / 'yolo' / 'best.pt',
        'type': 'yolo',
        'size': 6,
        'fps': 45
    },
    'mobilenetv2': {
        'name': 'MobileNetV2',
        'path': models_dir / 'mobilenetv2' / 'best_model.keras',
        'type': 'tensorflow',
        'size': 14,
        'fps': 25
    },
    'cnn': {
        'name': 'CNN',
        'path': models_dir / 'cnn' / 'best_model.keras',
        'type': 'tensorflow',
        'size': 125,
        'fps': 12
    }
}

for key, info in model_info.items():
    if info['path'].exists():
        available_models[key] = info
        print(f"   ✅ {info['name']} available at: {info['path']}")

if not available_models:
    print("\n❌ Tidak ada model yang tersedia!")
    print("   Pastikan training sudah selesai.")
    raise SystemExit("No model available")

# Pilih model terbaik (prioritas: YOLO > MobileNetV2 > CNN)
priority = ['yolo', 'mobilenetv2', 'cnn']
selected_model = None
for key in priority:
    if key in available_models:
        selected_model = key
        break

print(f"\n✅ Model terpilih: {available_models[selected_model]['name']}")
print(f"   - Path: {available_models[selected_model]['path']}")
print(f"   - Size: {available_models[selected_model]['size']} MB")
print(f"   - Est. FPS: {available_models[selected_model]['fps']}")

# ============================================
# 3. LOAD MODEL
# ============================================

print("\n[Loading Model]")

model_type = available_models[selected_model]['type']
model_path = available_models[selected_model]['path']

if model_type == 'yolo':
    from ultralytics import YOLO
    model = YOLO(str(model_path))
    print(f"✅ YOLO model loaded")
    
elif model_type == 'tensorflow':
    import tensorflow as tf
    from tensorflow.keras.models import load_model
    model = load_model(str(model_path))
    print(f"✅ TensorFlow model loaded")
    
    # Load class names
    class_names = ['Awake', 'Drowsy']

# ============================================
# 4. REAL-TIME DETECTION FUNCTIONS
# ============================================

print("\n[Setting Up Detection]")

# Counter for drowsy detection
drowsy_counter = 0
alert_threshold = 5  # Number of consecutive drowsy frames before alert
frame_counter = 0
fps = 0
start_time = time.time()

def predict_yolo(frame, model):
    """Prediksi dengan YOLO"""
    results = model(frame, imgsz=224, verbose=False)
    
    if results and len(results) > 0:
        probs = results[0].probs
        if probs is not None:
            pred_class = probs.top1
            confidence = probs.top1conf.item()
            
            if pred_class == 1:  # Drowsy
                return 'Drowsy', confidence
            else:
                return 'Awake', confidence
    
    return 'Awake', 0.5

def predict_tensorflow(frame, model):
    """Prediksi dengan TensorFlow"""
    # Preprocess frame
    img = cv2.resize(frame, (224, 224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    
    # Predict
    prediction = model.predict(img, verbose=False)
    confidence = float(prediction[0][0])
    
    if confidence > 0.5:
        return 'Drowsy', confidence
    else:
        return 'Awake', 1 - confidence

def draw_info(frame, status, confidence, fps, drowsy_count, alert):
    """Draw information on frame"""
    h, w = frame.shape[:2]
    
    # Background untuk info
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 120), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)
    
    # Status
    status_color = (0, 255, 0) if status == 'Awake' else (0, 0, 255)
    cv2.putText(frame, f"Status: {status}", (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, status_color, 3)
    
    # Confidence
    cv2.putText(frame, f"Confidence: {confidence:.2%}", (20, 75), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    # FPS
    cv2.putText(frame, f"FPS: {fps:.1f}", (w - 150, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    # Drowsy counter
    cv2.putText(frame, f"Drowsy Count: {drowsy_count}", (w - 200, 75), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    # Alert
    if alert:
        # Red border
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 255), 10)
        # Alert text
        cv2.putText(frame, "⚠️ ALERT: DROWSY DRIVER!", (w//2 - 200, h//2), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)
    
    return frame

def save_alert_frame(frame, status, confidence):
    """Save frame when drowsy detected"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = results_dir / f"alert_{timestamp}_{confidence:.2f}.jpg"
    cv2.imwrite(str(filename), frame)
    print(f"   📸 Alert saved: {filename}")

# ============================================
# 5. REAL-TIME DETECTION
# ============================================

print("\n" + "="*60)
print("🎥 STARTING REAL-TIME DETECTION")
print("="*60)
print("\n📌 Instructions:")
print("   - Press 'q' to quit")
print("   - Press 's' to save current frame")
print("   - Press 'r' to reset drowsy counter")
print("   - Press 'p' to pause/resume")
print("\n" + "="*60)

def run_real_time_detection():
    """Main real-time detection loop"""
    global drowsy_counter, frame_counter, fps, start_time
    
    # Initialize webcam
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Webcam tidak dapat diakses!")
        print("   Pastikan webcam terhubung dan tidak digunakan aplikasi lain.")
        return
    
    # Get frame size
    ret, frame = cap.read()
    if not ret:
        print("❌ Gagal membaca dari webcam!")
        cap.release()
        return
    
    print(f"✅ Webcam initialized: {frame.shape[1]}x{frame.shape[0]}")
    
    paused = False
    frame_count = 0
    drowsy_frames = 0
    alert_triggered = False
    
    while True:
        if not paused:
            # Read frame
            ret, frame = cap.read()
            if not ret:
                print("❌ Gagal membaca frame!")
                break
            
            frame_counter += 1
            
            # Predict
            if model_type == 'yolo':
                status, confidence = predict_yolo(frame, model)
            else:
                status, confidence = predict_tensorflow(frame, model)
            
            # Update drowsy counter
            if status == 'Drowsy':
                drowsy_frames += 1
                drowsy_counter += 1
            else:
                drowsy_frames = 0
            
            # Check alert
            alert = drowsy_frames >= alert_threshold
            if alert and not alert_triggered:
                alert_triggered = True
                save_alert_frame(frame, status, confidence)
            
            if not alert:
                alert_triggered = False
            
            # Calculate FPS
            frame_count += 1
            if frame_count % 30 == 0:
                current_time = time.time()
                fps = 30 / (current_time - start_time)
                start_time = current_time
            
            # Draw info
            display_frame = draw_info(frame.copy(), status, confidence, fps, drowsy_counter, alert)
            
        else:
            # Paused - show pause message
            display_frame = frame.copy()
            h, w = display_frame.shape[:2]
            cv2.putText(display_frame, "⏸️ PAUSED", (w//2 - 100, h//2), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 0), 4)
        
        # Show frame
        cv2.imshow('Driver Drowsiness Detection', display_frame)
        
        # Handle keyboard input
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            print("\n🛑 User requested quit")
            break
        elif key == ord('s'):
            # Save current frame
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = results_dir / f"capture_{timestamp}_{status}.jpg"
            cv2.imwrite(str(filename), frame)
            print(f"📸 Frame saved: {filename}")
        elif key == ord('r'):
            # Reset drowsy counter
            drowsy_counter = 0
            drowsy_frames = 0
            print("🔄 Drowsy counter reset")
        elif key == ord('p'):
            # Pause/Resume
            paused = not paused
            print(f"⏸️ {'Paused' if paused else 'Resumed'}")
    
    # Cleanup
    cap.release()
    cv2.destroyAllWindows()
    print("\n✅ Real-time detection stopped")

# ============================================
# 6. RUN DETECTION
# ============================================

print("\n[Starting Detection]")

try:
    run_real_time_detection()
except KeyboardInterrupt:
    print("\n🛑 Interrupted by user")
    cv2.destroyAllWindows()
except Exception as e:
    print(f"\n❌ Error: {e}")
    cv2.destroyAllWindows()

# ============================================
# 7. SUMMARY
# ============================================

print("\n" + "="*60)
print("📊 REAL-TIME DETECTION SUMMARY")
print("="*60)

print(f"\n📋 Informasi:")
print(f"   - Model: {available_models[selected_model]['name']}")
print(f"   - Model Type: {model_type}")
print(f"   - Alert Threshold: {alert_threshold} consecutive frames")
print(f"   - Results saved to: {results_dir}")

# Count saved alert images
alert_images = list(results_dir.glob('alert_*.jpg'))
capture_images = list(results_dir.glob('capture_*.jpg'))

print(f"\n📸 Saved Images:")
print(f"   - Alert images: {len(alert_images)}")
print(f"   - Capture images: {len(capture_images)}")

print("\n" + "="*60)
print("✅ REAL-TIME DETECTION COMPLETE!")
print("="*60)